In [ ]:
import datetime
import sklearn

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import KernelPCA
import numpy as np
import pandas as pd
import math
import keras
import matplotlib.pyplot as plt
import tensorflow as tf
tf.random.set_seed(2025)
df = pd.read_csv('final_data_adj.csv')
dates = df.Date
df = df.set_index('Date')

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled = scaler.fit_transform(df)
df_scaled = pd.DataFrame(df_scaled, columns=list(df.columns))

In [ ]:
def sampler(df, window):

    xRes = []
    yRes = []

    for i in range(0, len(df) - window):
        res = []

        for j in range(0, window):
            r = []
            for col in df.columns:
                r.append(df[col][i + j])
            res.append(r)
        xRes.append(res)
        yRes.append(df[['Open', 'Close']].iloc[i + window].values)
    return np.array(xRes), np.array(yRes)

In [ ]:
SPLIT = 0.8
(xVal, yVal) = sampler(df_scaled, 20)
X_train = xVal[:int(SPLIT * len(xVal))]
y_train = yVal[:int(SPLIT * len(yVal))]
X_test = xVal[int(SPLIT * len(xVal)):]
y_test = yVal[int(SPLIT * len(yVal)):]

In [ ]:
mv = keras.Sequential()
mv.add(keras.layers.LSTM(200, input_shape=(X_train.shape[1], X_train.shape[2])))
mv.add(keras.layers.Dropout(0.2))
mv.add(keras.layers.Dense(2, activation='linear'))
mv.compile(loss = 'MeanSquaredError', metrics=['MAE'], optimizer='Adam')
mv.summary()

In [ ]:
history = mv.fit(X_train, y_train, epochs=20)

In [ ]:
predicted_values = mv.predict(X_test)

d = {'Predicted_Open': predicted_values[:, 0],
'Predicted_Close': predicted_values[:,1],
'Actual_Open': y_test[:,0],
'Actual_Close': y_test[:,1]}

d = pd.DataFrame(d)
d.index = df.index[-len(y_test):]

fig, ax = plt.subplots(figsize=(15, 6))

highlight_start = int(len(d) * 0.9)
highlight_end = len(d) - 1

plt.plot(d[['Actual_Open', 'Actual_Close']][:highlight_start], label=['Actual_Open', 'Actual_Close'])
plt.plot(d[['Predicted_Open', 'Predicted_Close']], label=['Predicted_Open', 'Predicted_Close'], linestyle='--')
plt.axvspan(d.index[highlight_start], d.index[highlight_end], facecolor='lightgreen', alpha=0.5, label='Forecast')
plt.xticks(d.index[::50])
ax.legend()
plt.show()

In [ ]:
print('Predicted Open: MSE=', sklearn.metrics.mean_squared_error(d[f'Actual_Open'], d['Predicted_Open']))
print('Predicted Open: MAE=', sklearn.metrics.mean_absolute_error(d[f'Actual_Open'], d['Predicted_Open']))
print('Predicted Open: R2=', sklearn.metrics.r2_score(d[f'Actual_Open'], d['Predicted_Open']))
print('Predicted Close: MSE=', sklearn.metrics.mean_squared_error(d[f'Actual_Close'], d['Predicted_Close']))
print('Predicted Close: MAE=', sklearn.metrics.mean_absolute_error(d[f'Actual_Close'], d['Predicted_Close']))
print('Predicted Close: R2=', sklearn.metrics.r2_score(d[f'Actual_Close'], d['Predicted_Close']))

### PROPHET

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

df = pd.read_csv("monthly-car-sales.csv")
print(df)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df.Month,df.Sales)
plt.xticks(df.Month[::12])
plt.show()

In [ ]:
df.columns = ['ds','y']
df_data_train = df[df["ds"] < "1968-01"]
df_data_test = df[df["ds"] >= "1968-01"]

In [ ]:
from prophet import Prophet

model = Prophet(seasonality_mode='multiplicative',interval_width=0.85)
model.fit(df_data_train)

In [ ]:
df_future = model.make_future_dataframe(periods=12, freq='MS')
print(df_future)

In [ ]:
fore = model.predict(df_future)
print(fore[['ds', 'yhat', 'yhat_lower', 'yhat_upper']])

In [ ]:
forecast_plot = model.plot(fore)
axes = forecast_plot.gca()
axes.axvline(x=fore['ds'].iloc[-12], color='red', linestyle='--', label='Training End')

df_data_test['ds'] = pd.to_datetime(df_data_test['ds'])
plt.plot(df_data_test['ds'], df_data_test['y'],'ro', markersize=3, label='True Test Data')
plt.legend()
plt.show()